# MT Model Training - Sequential Fine-Tuning with SMaLL-100

This notebook tests whether **sequential fine-tuning** (similar language → baseline) improves MT performance compared to **direct fine-tuning** (baseline only).

## Model: SMaLL-100

- **Model:** `alirezamsh/small100` 
- **Size:** 256M parameters (smaller and faster than mBART-50's 610M)
- **Languages:** 100+ languages including Filipino languages
- **Architecture:** M2M-100 based
- **Advantage:** Lower memory usage, faster training, better for low-resource scenarios

## Experimental Design: Sequential Fine-Tuning

For each target language, we create TWO models:

### 1. Baseline Models (Direct Training)
- **Start:** SMaLL-100 (pretrained)
- **Train:** Distant language → Target language (e.g., `en→tl`)
- **Result:** Baseline Model

### 2. Experimental Models (Sequential Fine-Tuning)
- **Start:** SMaLL-100 (pretrained)
- **Step 1:** Train on Similar language → Target language (e.g., `bik→tl`)
- **Step 2:** **Continue training** from Step 1 on Distant language → Target language (e.g., `en→tl`)
- **Result:** Experimental Model (with similarity transfer)

### Three Target Languages
1. **Tagalog (tl)**
   - Baseline: `en→tl` only
   - Experimental: `bik→tl` THEN `en→tl`

2. **Ilonggo/Hiligaynon (hil)**
   - Baseline: `en→hil` only
   - Experimental: `msb→hil` THEN `en→hil`

3. **Waray (war)**
   - Baseline: `en→war` only
   - Experimental: `hil→war` THEN `en→war`

## Research Question
**Does "warming up" the model on a similar low-resource language first improve performance on the baseline task?**

Expected: `BLEU(Sequential) > BLEU(Baseline)`

## Install Required Packages

Run this cell first if packages are not installed.

In [ ]:
# Uncomment and run if needed
# !pip install transformers datasets evaluate sacrebleu torch sentencepiece accelerate

## Imports

In [ ]:
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch
from pathlib import Path
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

Set up training configurations for all language pairs.

In [ ]:
# Model configuration
MODEL_NAME = "alirezamsh/small100"
MAX_LENGTH = 128
BATCH_SIZE = 16  # Can use larger batch size due to smaller model (256M params)
LEARNING_RATE = 3e-5
NUM_EPOCHS_STAGE1 = 3  # For similar language training (Stage 1)
NUM_EPOCHS_STAGE2 = 3  # For baseline training (Stage 2)

# Directories
DATA_DIR = Path("../data/splits")
OUTPUT_DIR = Path("../models_small100")
LOGS_DIR = Path("../logs_small100")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Language code mapping for SMaLL-100
# SMaLL-100 uses tl (Tagalog) for Filipino languages
# Full list: https://huggingface.co/alirezamsh/small100#languages-covered
LANG_CODES = {
    "en": "en",      # English
    "tl": "tl",      # Tagalog (use for all Filipino languages)
    "bik": "tl",     # Bikolano → use Tagalog code
    "hil": "tl",     # Ilonggo → use Tagalog code
    "msb": "tl",     # Masbatenyo → use Tagalog code
    "war": "tl"      # Waray → use Tagalog code
}

# Experimental configurations
EXPERIMENTS = {
    "tagalog": {
        "target": "tl",
        "baseline_pair": "en-tl",
        "similar_pair": "bik-tl",
        "baseline_config": {
            "src_lang": "en",
            "tgt_lang": "tl",
        },
        "similar_config": {
            "src_lang": "tl",  # Bikolano → use tl code
            "tgt_lang": "tl",
        }
    },
    "ilonggo": {
        "target": "hil",
        "baseline_pair": "en-hil",
        "similar_pair": "msb-hil",
        "baseline_config": {
            "src_lang": "en",
            "tgt_lang": "tl",  # Ilonggo → use tl code
        },
        "similar_config": {
            "src_lang": "tl",  # Masbatenyo → use tl code
            "tgt_lang": "tl",
        }
    },
    "waray": {
        "target": "war",
        "baseline_pair": "en-war",
        "similar_pair": "hil-war",
        "baseline_config": {
            "src_lang": "en",
            "tgt_lang": "tl",  # Waray → use tl code
        },
        "similar_config": {
            "src_lang": "tl",  # Ilonggo → use tl code
            "tgt_lang": "tl",
        }
    }
}

print("Experimental Configuration:")
print(f"  Model: {MODEL_NAME} (256M parameters)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs (Stage 1 - Similar): {NUM_EPOCHS_STAGE1}")
print(f"  Epochs (Stage 2 - Baseline): {NUM_EPOCHS_STAGE2}")
print(f"  Max length: {MAX_LENGTH}")
print(f"\nTarget Languages: {len(EXPERIMENTS)}")
for lang, config in EXPERIMENTS.items():
    print(f"  - {lang.capitalize()}: {config['baseline_pair']} (baseline) vs {config['similar_pair']}→{config['baseline_pair']} (sequential)")

## Helper Functions

Functions to load data, preprocess, and evaluate models.

In [ ]:
def load_data_for_pair(pair_name):
    """
    Load train and dev splits for a language pair.
    
    Args:
        pair_name: Language pair (e.g., 'en-tl')
    
    Returns:
        DatasetDict with 'train' and 'validation' splits
    """
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    # Load train data
    with open(pair_dir / f"train.{src_code}", "r", encoding="utf-8") as f:
        train_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"train.{tgt_code}", "r", encoding="utf-8") as f:
        train_tgt = [line.strip() for line in f.readlines()]
    
    # Load dev data
    with open(pair_dir / f"dev.{src_code}", "r", encoding="utf-8") as f:
        dev_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"dev.{tgt_code}", "r", encoding="utf-8") as f:
        dev_tgt = [line.strip() for line in f.readlines()]
    
    # Create datasets
    train_dataset = Dataset.from_dict({
        "src": train_src,
        "tgt": train_tgt
    })
    
    dev_dataset = Dataset.from_dict({
        "src": dev_src,
        "tgt": dev_tgt
    })
    
    dataset_dict = DatasetDict({
        "train": train_dataset,
        "validation": dev_dataset
    })
    
    print(f"Loaded {pair_name}:")
    print(f"  Train: {len(train_dataset)} examples")
    print(f"  Dev: {len(dev_dataset)} examples")
    
    return dataset_dict


def create_preprocess_function(tokenizer, src_lang, tgt_lang, max_length):
    """
    Create a preprocessing function for tokenization.
    
    SMaLL-100 uses M2M100 tokenizer which handles language codes differently.
    """
    def preprocess(batch):
        # Tokenize source
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch["src"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        
        # Tokenize target
        tokenizer.src_lang = tgt_lang  # For M2M100, set src_lang for target
        labels = tokenizer(
            batch["tgt"],
            truncation=True,
            padding="max_length",
            max_length=max_length
        )["input_ids"]
        
        inputs["labels"] = labels
        return inputs
    
    return preprocess


def create_compute_metrics(tokenizer):
    """
    Create a function to compute BLEU score during evaluation.
    """
    bleu = evaluate.load("sacrebleu")
    
    def compute_metrics(eval_pred):
        preds, labels = eval_pred
        
        # Decode predictions
        if isinstance(preds, tuple):
            preds = preds[0]
        
        # Replace -100 in labels (used for padding)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        
        # Decode
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        # Compute BLEU
        result = bleu.compute(
            predictions=decoded_preds,
            references=[[label] for label in decoded_labels]
        )
        
        return {"bleu": result["score"]}
    
    return compute_metrics

print("✓ Helper functions defined")

## Train a Single Stage

Function to train one language pair (can be baseline or Stage 1/2).

In [ ]:
def train_single_stage(pair_name, config, model_name_or_path, output_subdir, num_epochs, stage_name=""):
    """
    Train an MT model for a single stage.
    
    Args:
        pair_name: Language pair (e.g., 'en-tl', 'bik-tl')
        config: Configuration dictionary with 'src_lang' and 'tgt_lang'
        model_name_or_path: Either MODEL_NAME or path to previously trained model
        output_subdir: Subdirectory name for saving
        num_epochs: Number of epochs to train
        stage_name: Description for logging
    
    Returns:
        Training results, metrics, and path to saved model
    """
    print("\n" + "=" * 80)
    print(f"Training: {pair_name.upper()}")
    if stage_name:
        print(f"Stage: {stage_name}")
    print("=" * 80)
    
    # Load tokenizer and model
    print("\n1. Loading model and tokenizer...")
    tokenizer = M2M100Tokenizer.from_pretrained(MODEL_NAME)
    model = M2M100ForConditionalGeneration.from_pretrained(model_name_or_path)
    print(f"   Loaded from: {model_name_or_path}")
    print(f"   Model size: ~256M parameters")
    
    # Load dataset
    print("\n2. Loading dataset...")
    dataset = load_data_for_pair(pair_name)
    
    # Tokenize dataset
    print("\n3. Tokenizing dataset...")
    preprocess_fn = create_preprocess_function(
        tokenizer,
        config['src_lang'],
        config['tgt_lang'],
        MAX_LENGTH
    )
    tokenized_dataset = dataset.map(preprocess_fn, batched=True)
    
    # Set up training arguments
    output_dir = OUTPUT_DIR / output_subdir
    
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=num_epochs,
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=True,
        logging_dir=str(LOGS_DIR / output_subdir),
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    
    # Create trainer
    print("\n4. Setting up trainer...")
    compute_metrics_fn = create_compute_metrics(tokenizer)
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_fn,
    )
    
    # Train
    print("\n5. Starting training...")
    train_result = trainer.train()
    
    # Save final model
    print("\n6. Saving model...")
    final_model_path = output_dir / "final_model"
    trainer.save_model(str(final_model_path))
    tokenizer.save_pretrained(str(final_model_path))
    
    # Evaluate on dev set
    print("\n7. Final evaluation on dev set...")
    eval_results = trainer.evaluate()
    
    # Save results
    results = {
        "pair": pair_name,
        "stage": stage_name,
        "model_source": model_name_or_path,
        "config": config,
        "train_results": {
            "train_loss": train_result.training_loss,
            "train_runtime": train_result.metrics["train_runtime"],
            "train_samples_per_second": train_result.metrics["train_samples_per_second"],
        },
        "eval_results": eval_results
    }
    
    results_file = output_dir / "training_results.json"
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ Training complete")
    print(f"  Final BLEU: {eval_results['eval_bleu']:.2f}")
    print(f"  Model saved to: {final_model_path}")
    print(f"  Results saved to: {results_file}")
    
    return results, str(final_model_path)


def train_experiment(target_lang_name, experiment_config):
    """
    Run complete experiment for one target language:
    1. Train baseline model (distant → target)
    2. Train experimental model Stage 1 (similar → target)
    3. Train experimental model Stage 2 (continue with distant → target)
    
    Args:
        target_lang_name: Name of target language (e.g., 'tagalog')
        experiment_config: Configuration dictionary from EXPERIMENTS
    
    Returns:
        Dictionary with all results
    """
    print("\n" + "#" * 80)
    print(f"# EXPERIMENT: {target_lang_name.upper()}")
    print("#" * 80)
    
    baseline_pair = experiment_config['baseline_pair']
    similar_pair = experiment_config['similar_pair']
    
    all_results = {}
    
    # BASELINE MODEL
    print(f"\n{'='*80}")
    print(f"BASELINE MODEL: {baseline_pair}")
    print(f"Training SMaLL-100 directly on {baseline_pair} data")
    print(f"{'='*80}")
    
    baseline_results, baseline_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_baseline",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Baseline (Direct Training)"
    )
    all_results['baseline'] = baseline_results
    
    # EXPERIMENTAL MODEL - STAGE 1
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL MODEL - STAGE 1: {similar_pair}")
    print(f"Training SMaLL-100 on similar language data ({similar_pair})")
    print(f"{'='*80}")
    
    stage1_results, stage1_model_path = train_single_stage(
        pair_name=similar_pair,
        config=experiment_config['similar_config'],
        model_name_or_path=MODEL_NAME,
        output_subdir=f"{target_lang_name}_experimental_stage1",
        num_epochs=NUM_EPOCHS_STAGE1,
        stage_name="Stage 1: Similar Language"
    )
    all_results['experimental_stage1'] = stage1_results
    
    # EXPERIMENTAL MODEL - STAGE 2
    print(f"\n{'='*80}")
    print(f"EXPERIMENTAL MODEL - STAGE 2: {baseline_pair}")
    print(f"Continuing from Stage 1 model, now training on {baseline_pair} data")
    print(f"{'='*80}")
    
    stage2_results, stage2_model_path = train_single_stage(
        pair_name=baseline_pair,
        config=experiment_config['baseline_config'],
        model_name_or_path=stage1_model_path,  # Load from Stage 1!
        output_subdir=f"{target_lang_name}_experimental_stage2",
        num_epochs=NUM_EPOCHS_STAGE2,
        stage_name="Stage 2: Baseline (After Similar)"
    )
    all_results['experimental_stage2'] = stage2_results
    
    # SUMMARY
    print("\n" + "=" * 80)
    print(f"EXPERIMENT COMPLETE: {target_lang_name.upper()}")
    print("=" * 80)
    print(f"\nBaseline Model ({baseline_pair} only):")
    print(f"  BLEU: {all_results['baseline']['eval_results']['eval_bleu']:.2f}")
    print(f"\nExperimental Model ({similar_pair} → {baseline_pair}):")
    print(f"  Stage 1 ({similar_pair}): {all_results['experimental_stage1']['eval_results']['eval_bleu']:.2f}")
    print(f"  Stage 2 ({baseline_pair}): {all_results['experimental_stage2']['eval_results']['eval_bleu']:.2f}")
    
    improvement = all_results['experimental_stage2']['eval_results']['eval_bleu'] - all_results['baseline']['eval_results']['eval_bleu']
    print(f"\nImprovement: {improvement:+.2f} BLEU points")
    
    if improvement > 0:
        print("✓ Sequential fine-tuning IMPROVED performance")
    elif improvement < 0:
        print("✗ Sequential fine-tuning DEGRADED performance")
    else:
        print("= No difference in performance")
    
    # Save experiment summary
    summary_file = OUTPUT_DIR / f"{target_lang_name}_experiment_summary.json"
    with open(summary_file, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
    
    return all_results

print("✓ Training functions defined")

## Test Set Evaluation Functions

Functions to evaluate trained models on the held-out test set for final results.

In [ ]:
def load_test_data(pair_name):
    """
    Load test set for a language pair.
    """
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    with open(pair_dir / f"test.{src_code}", "r", encoding="utf-8") as f:
        test_src = [line.strip() for line in f.readlines()]
    
    with open(pair_dir / f"test.{tgt_code}", "r", encoding="utf-8") as f:
        test_tgt = [line.strip() for line in f.readlines()]
    
    print(f"Loaded test set for {pair_name}: {len(test_src)} examples")
    return test_src, test_tgt


def evaluate_model_on_test(model_path, pair_name, config, batch_size=16):
    """
    Evaluate a trained model on the test set.
    """
    print(f"\n{'='*80}")
    print(f"Evaluating model on TEST SET: {pair_name}")
    print(f"Model: {model_path}")
    print(f"{'='*80}")
    
    # Load model and tokenizer
    print("\n1. Loading model and tokenizer...")
    tokenizer = M2M100Tokenizer.from_pretrained(MODEL_NAME)
    model = M2M100ForConditionalGeneration.from_pretrained(model_path)
    model.eval()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"   Model loaded on: {device}")
    
    # Load test data
    print("\n2. Loading test data...")
    test_src, test_tgt = load_test_data(pair_name)
    
    # Generate translations
    print("\n3. Generating translations...")
    tokenizer.src_lang = config['src_lang']
    
    all_predictions = []
    
    for i in range(0, len(test_src), batch_size):
        batch_src = test_src[i:i + batch_size]
        
        inputs = tokenizer(
            batch_src,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)
        
        # Generate with forced target language
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.get_lang_id(config['tgt_lang']),
                max_length=MAX_LENGTH,
                num_beams=5,
                early_stopping=True
            )
        
        batch_predictions = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )
        all_predictions.extend(batch_predictions)
        
        if (i // batch_size + 1) % 10 == 0:
            print(f"   Processed {i + len(batch_src)}/{len(test_src)} examples...")
    
    print(f"   ✓ Generated {len(all_predictions)} translations")
    
    # Compute BLEU score
    print("\n4. Computing BLEU score...")
    bleu = evaluate.load("sacrebleu")
    result = bleu.compute(
        predictions=all_predictions,
        references=[[ref] for ref in test_tgt]
    )
    
    bleu_score = result["score"]
    print(f"   Test Set BLEU: {bleu_score:.2f}")
    
    # Get samples
    num_samples = min(5, len(test_src))
    samples = []
    for i in range(num_samples):
        samples.append({
            "source": test_src[i],
            "reference": test_tgt[i],
            "prediction": all_predictions[i]
        })
    
    return {
        "bleu": bleu_score,
        "num_examples": len(test_src),
        "samples": samples
    }


def evaluate_experiment_on_test(target_lang_name, experiment_config):
    """
    Evaluate both baseline and experimental models on test set.
    """
    print("\n" + "#" * 80)
    print(f"# TEST SET EVALUATION: {target_lang_name.upper()}")
    print("#" * 80)
    
    baseline_pair = experiment_config['baseline_pair']
    baseline_config = experiment_config['baseline_config']
    
    results = {}
    
    # Evaluate baseline
    baseline_model_path = OUTPUT_DIR / f"{target_lang_name}_baseline" / "final_model"
    if not baseline_model_path.exists():
        print(f"\n❌ Baseline model not found: {baseline_model_path}")
        return None
    
    results['baseline'] = evaluate_model_on_test(
        str(baseline_model_path),
        baseline_pair,
        baseline_config
    )
    
    # Evaluate experimental
    experimental_model_path = OUTPUT_DIR / f"{target_lang_name}_experimental_stage2" / "final_model"
    if not experimental_model_path.exists():
        print(f"\n❌ Experimental model not found: {experimental_model_path}")
        return None
    
    results['experimental'] = evaluate_model_on_test(
        str(experimental_model_path),
        baseline_pair,
        baseline_config
    )
    
    # Compare
    print("\n" + "=" * 80)
    print(f"TEST SET RESULTS: {target_lang_name.upper()}")
    print("=" * 80)
    print(f"\nBaseline Model ({baseline_pair} only):")
    print(f"  Test BLEU: {results['baseline']['bleu']:.2f}")
    print(f"\nExperimental Model (Sequential Fine-tuning):")
    print(f"  Test BLEU: {results['experimental']['bleu']:.2f}")
    
    improvement = results['experimental']['bleu'] - results['baseline']['bleu']
    print(f"\nTest Set Improvement: {improvement:+.2f} BLEU points")
    
    if improvement > 0:
        print("✓ Sequential fine-tuning IMPROVED performance")
    elif improvement < 0:
        print("✗ Sequential fine-tuning DEGRADED performance")
    else:
        print("= No difference")
    
    # Show samples
    print("\n" + "-" * 80)
    print("Sample Translations (first 3):")
    print("-" * 80)
    for i in range(min(3, len(results['baseline']['samples']))):
        sample = results['baseline']['samples'][i]
        print(f"\n[Example {i+1}]")
        print(f"Source:      {sample['source']}")
        print(f"Reference:   {sample['reference']}")
        print(f"Baseline:    {sample['prediction']}")
        print(f"Experimental: {results['experimental']['samples'][i]['prediction']}")
    
    # Save results
    test_results_file = OUTPUT_DIR / f"{target_lang_name}_test_results.json"
    save_results = {
        "baseline_bleu": results['baseline']['bleu'],
        "experimental_bleu": results['experimental']['bleu'],
        "improvement": improvement,
        "baseline_samples": results['baseline']['samples'],
        "experimental_samples": results['experimental']['samples']
    }
    with open(test_results_file, "w") as f:
        json.dump(save_results, f, indent=2)
    
    print(f"\n✓ Test results saved to: {test_results_file}")
    return results

print("✓ Test evaluation functions defined")

## Run Single Experiment

Test with one target language first to verify the setup.

In [ ]:
# Run Tagalog experiment (uncomment to run)
# target_lang = "tagalog"
# results = train_experiment(target_lang, EXPERIMENTS[target_lang])

## Evaluate Single Experiment on Test Set

In [ ]:
# Evaluate on test set (run after training)
# target_lang = "tagalog"
# test_results = evaluate_experiment_on_test(target_lang, EXPERIMENTS[target_lang])

## Run All Experiments

Run all three experiments. This will train 9 models total (3 per language).

**Expected time with SMaLL-100:**
- **RTX 2050 (4GB):** ~30-45 min per experiment = 1.5-2.5 hours total
- **RTX 3050/4050:** ~20-30 min per experiment = 1-1.5 hours total

Much faster than mBART-50 due to smaller model size!

In [ ]:
# Run all experiments (uncomment to run)
# all_experiment_results = {}
# 
# for target_lang, exp_config in EXPERIMENTS.items():
#     try:
#         results = train_experiment(target_lang, exp_config)
#         all_experiment_results[target_lang] = results
#     except Exception as e:
#         print(f"\n❌ Error in {target_lang} experiment: {e}")
#         import traceback
#         traceback.print_exc()
#         continue
# 
# # Save overall summary
# final_summary_file = OUTPUT_DIR / "all_experiments_summary.json"
# with open(final_summary_file, "w") as f:
#     json.dump(all_experiment_results, f, indent=2)
# 
# print("\n" + "#" * 80)
# print("# ALL EXPERIMENTS COMPLETE")
# print("#" * 80)
# print(f"\nFinal summary saved to: {final_summary_file}")
# 
# # Print comparison table
# print("\n" + "=" * 80)
# print("RESULTS SUMMARY (Dev Set)")
# print("=" * 80)
# print(f"{'Target Language':<20} {'Baseline BLEU':<15} {'Sequential BLEU':<15} {'Improvement':<15}")
# print("-" * 80)
# for target_lang, results in all_experiment_results.items():
#     baseline_bleu = results['baseline']['eval_results']['eval_bleu']
#     sequential_bleu = results['experimental_stage2']['eval_results']['eval_bleu']
#     improvement = sequential_bleu - baseline_bleu
#     print(f"{target_lang.capitalize():<20} {baseline_bleu:<15.2f} {sequential_bleu:<15.2f} {improvement:+.2f}")

## Evaluate All Experiments on Test Set

Run final evaluation on test set for all trained models.

In [ ]:
# Evaluate all experiments on test set (uncomment after training)
# all_test_results = {}
# 
# for target_lang, exp_config in EXPERIMENTS.items():
#     try:
#         test_results = evaluate_experiment_on_test(target_lang, exp_config)
#         if test_results:
#             all_test_results[target_lang] = test_results
#     except Exception as e:
#         print(f"\n❌ Error evaluating {target_lang}: {e}")
#         import traceback
#         traceback.print_exc()
#         continue
# 
# # Save test results
# test_summary_file = OUTPUT_DIR / "all_experiments_test_results.json"
# with open(test_summary_file, "w") as f:
#     json.dump(all_test_results, f, indent=2)
# 
# print("\n" + "#" * 80)
# print("# TEST SET EVALUATION COMPLETE")
# print("#" * 80)
# print(f"\nTest results saved to: {test_summary_file}")
# 
# # Print final comparison
# print("\n" + "=" * 80)
# print("FINAL TEST SET RESULTS")
# print("=" * 80)
# print(f"{'Target Language':<20} {'Baseline BLEU':<15} {'Sequential BLEU':<15} {'Improvement':<15}")
# print("-" * 80)
# for target_lang, results in all_test_results.items():
#     baseline_bleu = results['baseline']['bleu']
#     sequential_bleu = results['experimental']['bleu']
#     improvement = sequential_bleu - baseline_bleu
#     print(f"{target_lang.capitalize():<20} {baseline_bleu:<15.2f} {sequential_bleu:<15.2f} {improvement:+.2f}")
# 
# # Calculate average
# if all_test_results:
#     improvements = [results['experimental']['bleu'] - results['baseline']['bleu'] 
#                     for results in all_test_results.values()]
#     avg_improvement = sum(improvements) / len(improvements)
#     print("-" * 80)
#     print(f"{'Average Improvement':<20} {'':<15} {'':<15} {avg_improvement:+.2f}")

## Summary and Comparison

### SMaLL-100 vs mBART-50

| Feature | SMaLL-100 | mBART-50 |
|---------|-----------|----------|
| **Parameters** | 256M | 610M |
| **Memory Usage** | ~2-3 GB VRAM | ~4-6 GB VRAM |
| **Training Speed** | ~2x faster | Baseline |
| **Batch Size** | 16 (RTX 2050) | 8 (RTX 2050) |
| **Languages** | 100+ | 50 |
| **Architecture** | M2M-100 | mBART |
| **Best For** | Low-resource, fast experiments | High-resource, max quality |

### Advantages of SMaLL-100 for This Project:

1. **Faster Training:** ~2x faster per epoch due to smaller model
2. **Lower Memory:** Can use larger batch sizes (16 vs 8)
3. **Same Architecture:** M2M-100 based, designed for multilingual MT
4. **Low-Resource Friendly:** Optimized for limited data scenarios
5. **More Languages:** Covers 100+ languages including Filipino

### Expected Performance:

SMaLL-100 may achieve slightly lower absolute BLEU scores than mBART-50 due to smaller capacity, but:
- **Relative improvements** from sequential fine-tuning should be similar or better
- **Faster iteration** allows more experimentation
- **Lower resource requirements** make it more practical for research

### Model Output Structure:
```
models_small100/
  ├── tagalog_baseline/
  ├── tagalog_experimental_stage1/
  ├── tagalog_experimental_stage2/
  ├── ilonggo_baseline/
  ├── ilonggo_experimental_stage1/
  ├── ilonggo_experimental_stage2/
  ├── waray_baseline/
  ├── waray_experimental_stage1/
  ├── waray_experimental_stage2/
  └── all_experiments_summary.json
```

### Tips:

1. **Start with one experiment** to verify setup
2. **Monitor training speed** - should be ~2x faster than mBART
3. **Compare results** between SMaLL-100 and mBART-50
4. **Use test set** for final evaluation and comparison
5. **Report both models** in your paper for completeness